In [1]:
import pandas as pd

# Load the cleaned tickets CSV
df = pd.read_parquet("../data/raw/complaint_embeddings.parquet")

In [2]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


c:\Users\dagic\OneDrive\Documents\KAIM\Week_7\rag-complaint-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 938.83it/s]


In [3]:
# If embeddings already exist in the Parquet file, stack them into a numpy array instantly
embeddings = np.vstack(df['embedding'].values).astype("float32")

# Normalize them if you want to use IndexFlatIP (Inner Product/Cosine Similarity)
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)


In [4]:
def retrieve(question, top_k=3):
    """Find the most relevant complaints for a question."""
    q_vec = embed_model.encode(question, normalize_embeddings=True).astype("float32").reshape(1, -1)
    scores, indices = index.search(q_vec, top_k)
    
    # FIX: Use .iloc with the entire list of matching integer positions from FAISS
    results_df = df.iloc[indices[0]]
    
    # Convert those matching rows to dictionaries so your loop 'for i, c in enumerate(retrieved, 1):' works perfectly!
    return results_df.to_dict(orient="records")

In [5]:
import os
from huggingface_hub import InferenceClient

HF_TOKEN = os.getenv("HF_TOKEN")

MODEL = "Qwen/Qwen2.5-7B-Instruct"

In [6]:
# Connect to HuggingFace Inference API — no download, runs on HF servers
client = InferenceClient(
    model=MODEL,
    token=HF_TOKEN,
)


def ask(prompt, max_new_tokens=200):
    """
    Send a prompt to the Qwen model via HuggingFace Inference API.
    Returns the answer as a plain string.

    No model is loaded locally — the request goes to HuggingFace's servers.
    """
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
    )
    return response.choices[0].message.content.strip()


In [7]:
# The complete RAG prompt template
RAG_PROMPT_TEMPLATE = """You are a customer support analyst at CrediTrust.
Use ONLY the ticket excerpts provided below to answer the question.
If the excerpts do not contain enough information to answer, say: "I don't have enough information to answer that."
Do not add any information not present in the excerpts.

Support ticket excerpts:
{context}

Question: {question}

Answer (based only on the excerpts above):"""


def rag_answer(question, top_k=3):
    """
    Full RAG pipeline:
      1. Retrieve top-k relevant tickets (semantic search)
      2. Build the prompt with those tickets as context
      3. Send to LLM and get answer
    """
    retrieved    = retrieve(question, top_k=top_k)
    formatted    = []
    
    for i, c in enumerate(retrieved, 1):
        # Safely extract nested metadata parameters
        meta = c.get('metadata', {})
        product = meta.get('product_category', 'Unknown Product')
        company = meta.get('company', 'Unknown Company')
        chunk_idx = meta.get('chunk_index', 0)
        
        # Pull the text content from your 'document' key
        text_content = c.get('document', '')
        
        # Append using your precise Parquet schema parameters
        formatted.append(
            f"[Excerpt {i} | Product: {product} | Company: {company} | Chunk: {chunk_idx}]\n{text_content}"
        )
        
    context_block = "\n\n".join(formatted)
    prompt        = RAG_PROMPT_TEMPLATE.format(context=context_block, question=question)
    answer        = ask(prompt)

    return {"question": question, "answer": answer, "sources": retrieved, "prompt": prompt}


def show_rag_result(result, show_prompt=False):
    print("=" * 65)
    print(f"QUESTION: {result['question']}")
    print("=" * 65)
    print(f"\nANSWER:\n{result['answer']}")
    print("\nSOURCES USED:")
    for s in result["sources"]:
        meta = s.get('metadata', {})
        product = meta.get('product_category', 'Unknown')
        company = meta.get('company', 'Unknown')
        chunk_idx = meta.get('chunk_index', '0')
        print(f"  - #{s['id']} | {product} | {company} | Chunk: {chunk_idx}")
    if show_prompt:
        print("\nFULL PROMPT SENT TO LLM:")
        print("-" * 40)
        print(result["prompt"])
    print("=" * 65)


print("RAG pipeline ready!")
print(f"Searching over {len(df)} support tickets")

RAG pipeline ready!
Searching over 1375327 support tickets


In [8]:
# QUESTION 1 — technical issue
result1 = rag_answer(
    "What are the most common reasons customers report credit card fraud?")
show_rag_result(result1)

QUESTION: What are the most common reasons customers report credit card fraud?

ANSWER:
The most common reasons customers report credit card fraud based on the provided excerpts include:
- Unauthorized purchases and transactions
- Multiple addresses being used for order and delivery of goods, which were not the customer's addresses
- Cards being reported as stolen
- Accounts being opened without the customer's knowledge or consent

SOURCES USED:
  - #5427299_1 | Credit Card | UPGRADE, INC. | Chunk: 1
  - #5684687_1 | Savings Account | CAPITAL ONE FINANCIAL CORPORATION | Chunk: 1
  - #11745532_0 | Money Transfer | Block, Inc. | Chunk: 0


In [9]:
# QUESTION 1 — technical issue
result1 = rag_answer(
    "Are customers experiencing unexpected fees when transferring money?")
show_rag_result(result1)

QUESTION: Are customers experiencing unexpected fees when transferring money?

ANSWER:
Based on the excerpts provided, customers are experiencing unexpected fees when transferring money, as indicated in the first excerpt regarding Money Transfer from JPMORGAN CHASE & CO.

SOURCES USED:
  - #8568894_1 | Money Transfer | JPMORGAN CHASE & CO. | Chunk: 1
  - #7092495_4 | Credit Card | Block, Inc. | Chunk: 4
  - #4970280_2 | Savings Account | JPMORGAN CHASE & CO. | Chunk: 2


In [10]:
# QUESTION 1 — technical issue
result1 = rag_answer(
    "What complaints exist regarding delayed processing times for Personal Loans?")
show_rag_result(result1)

QUESTION: What complaints exist regarding delayed processing times for Personal Loans?

ANSWER:
The complaint regarding delayed processing times for Personal Loans is that U.S. BANCORP's investigation took much longer than the required 30 days and there was no communication about the status of the investigation or that it would take longer than required by law.

SOURCES USED:
  - #3248956_4 | Credit Card | CITIBANK, N.A. | Chunk: 4
  - #12740470_1 | Personal Loan | Revolut Technologies Inc. | Chunk: 1
  - #5248649_1 | Personal Loan | U.S. BANCORP | Chunk: 1
